# 1. 뉴스 가져오기

## 1.1. 네이트 뉴스 페이지에서 기사 가져오기

In [2]:
from wrapper.news_fetcher import NewsFetcher
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *


news = NewsFetcher()

news_list = news.fetch_news(n_pages=5)
news.save_csv()
    
news_tags = [news.tags[i][0] for i in news.tags.keys()]
news_ids = {tag: [] for tag in news_tags}

for tag in news_tags:
    news.news[tag].pop(0)

news = news.news


 20%|██        | 1/5 [01:33<06:12, 93.06s/it]

정치


 40%|████      | 2/5 [03:05<04:38, 92.68s/it]

경제


 60%|██████    | 3/5 [04:35<03:02, 91.37s/it]

사회


 80%|████████  | 4/5 [06:12<01:33, 93.74s/it]

국제


100%|██████████| 5/5 [07:45<00:00, 93.20s/it]

과학/기술
정치
경제
사회
국제
과학/기술


## 1.2. failback: csv 파일로부터 뉴스 데이터 가져오기

In [3]:
from wrapper.news_fetcher import NewsFetcher


news = NewsFetcher().load_csv()

정치
경제
사회
국제
과학/기술


# 2. 뉴스 업로드

## 2.1. 업로드

In [3]:
from wrapper.api_wrapper import ApiWrapper


api = ApiWrapper()
uploaded_news = api.upload_news(news)

  0%|          | 0/5 [00:00<?, ?it/s]

failed to upload news, 윤 대통령, '김건희 특검법' 3번째 거부권…야당 "뻔뻔하고 가증스러"	{"result":false,"data":[{"exception":"com.rrkim.core.common.exception.UnhandledExecutionException","errorCode":"news.uploadFileError","description":"업로드된 파일이 문제 있습니다.","message":"요청을 처리하는 중에 오류가 발생하였습니다."}],"resultCount":1}


 20%|██        | 1/5 [03:54<15:39, 234.88s/it]

failed to upload news, 한은, 2024년 '화폐사랑 콘텐츠 공모전' 수상작 선정·포상	{"result":false,"data":[{"exception":"com.rrkim.core.common.exception.UnhandledExecutionException","errorCode":"news.uploadFileError","description":"업로드된 파일이 문제 있습니다.","message":"요청을 처리하는 중에 오류가 발생하였습니다."}],"resultCount":1}


100%|██████████| 5/5 [19:46<00:00, 237.39s/it]


## 2.2. failback: 뉴스 ID 복원

In [ ]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
on_server = api.download_news()
uploaded_news: list[UploadedNews] = []

for tag in news:
    for n in news[tag]:
        for o in on_server:
            if o["title"] == n.title:
                uploaded_news.append(
                    UploadedNews(
                        title=n.title,
                        content=n.content,
                        image=n.image,
                        press=n.press,
                        pub_time=n.pub_time,
                        tag=n.tag,
                        url=n.url,
                        id=o["newsIdx"],
                    )
                )
                break

len(uploaded_news)

api.save_csv(uploaded_news)

## 2.3. csv 파일로부터 업로드된 뉴스 불러오기

In [1]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
uploaded_news = api.load_csv()

# 3. 뉴스 요약

## 3.1. 뉴스 요약 진행

In [4]:
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *

import pickle


api = ApiWrapper()


llm = LLM(n_ctx=8192, max_tokens=1024)
summerized_news: list[SummerizedNews] = []

for news in tqdm(uploaded_news):
    llm.set_prompt(
        f"""
        [요청 사항]
        - 이 뉴스를 다음 양식을 준수하는 세 문장으로 요약해 주세요.

        [준수 사항]
        - 첫 번째 문장은 이 기사에서 다루는 핵심 사건을 설명하는 120자 내외의 완결된 문장이어야 합니다.
        - 두 번째 문장은 사건의 배경과 관련된 맥락을 설명하는 120자 내외의 완결된 문장이어야 합니다.
        - 세 번째 문장은 사건의 진행과 결과를 설명하는 120자 내외의 완결된 문장이어야 합니다.

        [참고 사항]
        - 요약하신 자료는 텍스트 임베딩을 거쳐 클러스터링 작업에 사용될 것입니다.
        - 이 뉴스는 {news.tag} 분야의 뉴스입니다.
        - 이 뉴스는 {news.pub_time} 시점에 게시되었습니다.

        [예시]
        1. 지난 23일 16시 경 광주 광산구 아파트 주차장에서 차량 4대를 들이받고 벤츠를 버린 운전자가 사건 발생 12시간 만에 경찰에 자진 출석했다.
        2. 사고 직후 운전자는 아무런 조치 없이 연락처와 벤츠를 남기고 도주했으며, 사고 12시간 40분 만에 같은 날 오후 6시쯤 경찰에 출석했다.
        3. 경찰은 A씨를 들이받은 차량을 수습하지 않은 채 도주한 혐의를 적용하여 조사하고 있으며, CCTV 등을 통해 운전 경로를 추적 수사할 방침이다.

        """
    )

    content = llm.generate(
        instruction=
        f"""
        [뉴스 제목]
        {news.title}
        [뉴스 내용]
        {news.content}
        """[:8191],
        reset_prompt=True
    )

    summerized_news.append(SummerizedNews(title=news.title, content=content, topics="", id=news.id))


with open("summerized.pkl", "wb") as f:
    pickle.dump(summerized_news, f)


def get_summerized_news(id: int) -> SummerizedNews:
    for i in summerized_news:
        if i.id == id:
            return i
    
    return None


for i in summerized_news[:5]:
    print(i.content, end="\n\n")

  0%|          | 0/498 [00:00<?, ?it/s]/home/gpp/.local/lib/python3.11/site-packages/llama_cpp/llama.py:1138: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
100%|██████████| 498/498 [39:02<00:00,  4.70s/it]

1. 김여정 북한 노동당 부부장은 최근 한국의 행위를 "더러운 행위"라고 비난했다.
2. 김여정의 발언은 한국과 북한 간 갈등이 심화되고 있는 상황에서 이루어진 것으로, 긴장의 고조를 보여준다.
3. 김여정의 발언은 한국 정부와 북한 간의 관계를 더욱 악화시키는 요인이 될 가능성이 크다.

1. 여야는 내달 10일 김건희 여사에 대한 특검법을 재의결하기로 결정했다.
2. 김건희 여사의 비위 의혹과 관련된 여야는 특검법을 통해 수사와 처벌을 강화하기로 합의했다.
3. 이에 따라 특검법은 내달 10일 국회 본회의에서 재의결될 예정이다.

1. 여야는 12월 10일 '김건희 특검법' 재표결을 결정했다.
2. 김건희 여사는 윤석열 대통령의 부인으로, 특검법은 윤 대통령의 부적절한 행위에 대한 조사와 관련된 것으로 여겨진다.
3. 경찰은 윤 대통령의 부적절한 행위에 대한 수사를 지속하며, 특검법의 통과를 통해 더 깊이 있는 조사를 원하고 있다.

1. 외교부 2차관은 사도광산 추도식에 대한 한국의 불참이 일본에 대한 강한 항의이자 유감 표명이라고 밝혔다.
2. 사도광산 추도식은 한일 간 합의 수준에 미치지 않는 추도식으로, 한국은 이를 강하게 항의하고 유감을 표명했다.
3. 외교부는 일본의 적반하장식 태도에도 무대응을 유지하며, 추도식 문제를 한일관계 전반에 미치지 않도록 관리하겠다고 밝혔다.

1. 김여정 북한 노동당 부부장은 26일 국경 인근에 남측이 보낸 대북 전단과 물품들이 떨어졌다고 주장하며 이를 규탄했다.
2. 김 부부장은 지난 17일 담화에서 대북 전단에 반발하며 "대가를 치르게 될 것"이라고 위협했으나, 이번에는 보복을 예고하지 않았다.
3. 김 부부장은 이번 사건을 강력히 규탄하며, 안전보위기관들이 구역봉쇄와 수색 및 수거, 처치 작업을 진행하고 있다고 밝혔다.



## 3.3. failback: 요약된 뉴스 불러오기

In [10]:
import pickle
from entity.entity import *


with open("summerized.pkl", "rb") as f:
    summerized_news = pickle.load(f)


def get_summerized_news(id: int) -> SummerizedNews:
    for i in summerized_news:
        if i.id == id:
            return i
    
    return None

# 4. 임베딩

# 6. 클러스터링

## 6.1. DBSCAN

In [5]:
# from wrapper.llm_wrapper import LLM
# model = LLM(embedding=True).model
from llama_cpp import Llama


del llm

MODEL_PATH = "/home/gpp/src/model/llama3-korean-bllossom-8b/llama-3-Korean-Bllossom-8B-Q4_K_M.gguf"
model = Llama(model_path=MODEL_PATH, embedding=True, verbose=False)
clusters = {}

In [6]:
import numpy as np
from tqdm import tqdm

firsts = [  ]
seconds = [ ]
thirds = [  ]

for texts in tqdm(summerized_news):
    try:
        first, second, third = list(filter(lambda x: x.strip() != '', texts.content.split('\n')))
    except:
        continue

    firsts.append(np.mean(model.embed(first, normalize=True), axis=0))
    seconds.append(np.mean(model.embed(second, normalize=True), axis=0))
    thirds.append(np.mean(model.embed(third, normalize=True), axis=0))

assert len(firsts) == len(seconds) == len(thirds)

first_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in firsts) - len(embedding)), 'constant') for embedding in firsts])
second_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in seconds) - len(embedding)), 'constant') for embedding in seconds])
third_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in thirds) - len(embedding)), 'constant') for embedding in thirds])

del model

100%|██████████| 498/498 [18:13<00:00,  2.20s/it]


In [7]:
texts.content.split("\n")

['1. 국방 혁신 AI 클라우드 도입이 가속화되면서 미국은 자국 민간 클라우드 서비스를 적극 활용하고 있으며, 국내에서도 공공기관이 민간 클라우드를 도입하는 사례가 늘고 있다.',
 '',
 '2. 국방 AI 클라우드 도입을 위해서는 민간 클라우드 사업자의 공공 클라우드 보안 인증을 받은 것이 필수적이며, 데이터의 민감도에 따라 폐쇄형 클라우드 또는 정부의 데이터센터 클라우드를 사용해야 한다.',
 '',
 '3. 국방 AI 클라우드 필수 조건으로는 데이터 주권, 운영 주권, 소프트웨어 주권이 필요하며, 이를 위해 클라우드 주권을 확보하고, 데이터 저장과 암호화 키 관리를 국내에서 스스로 관리해야 한다.']

In [8]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import Normalizer


norm = Normalizer()
first_embeddings = norm.fit_transform(first_embeddings)
second_embeddings = norm.fit_transform(second_embeddings)
third_embeddings = norm.fit_transform(third_embeddings)

first_similarity = pairwise_distances(first_embeddings)
second_similarity = pairwise_distances(second_embeddings)
third_similarity = pairwise_distances(third_embeddings)

# 세 유사도의 평균값을 최종 유사도로 사용
similarity_matrix = (first_similarity + second_similarity + third_similarity) / 3
metric="euclidean"

In [16]:
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity


dbscan = DBSCAN(eps=0.4, min_samples=4, metric="precomputed")
clusters_ = dbscan.fit_predict(similarity_matrix)

result_string = ""

# 클러스터 결과
print(len(set(clusters_)))
clusters = {}  # 클러스터를 저장할 딕셔너리

for cluster_id in set(clusters_):
    cluster_news_ids = set()  # 중복을 피하기 위해 set 사용

    if cluster_id != -1:  # -1은 노이즈
        cluster_string = f"[Cluster ID: {cluster_id}]\n"
        result_string += cluster_string
        print(cluster_string, end="")

        for i in np.where(clusters_ == cluster_id)[0]:
            # 중복된 ID를 추가하지 않도록 set에 추가
            clustered_string = f"\t{summerized_news[i].title}\n"
            result_string += clustered_string
            cluster_news_ids.add(summerized_news[i].id)
            print(clustered_string[:-1], summerized_news[i].id)

        clusters[cluster_id] = list(cluster_news_ids)

result_string

16
[Cluster ID: 0]
	외교 2차관 "사도광산 추도식 불참이 강한 항의이자 유감 표명" 4816
	한일 외교수장, G7계기 伊서 오늘 만날 듯…사도광산 논의 4819
	외교부 "추도식 불참 통보 때 유감 표명과 항의"…무대응 비판에 뒤늦게 수습 4833
	외교2차관 "사도광산 추도식 불참, 日에 강한 항의"(종합) 4895
	'우익 성향' 日산케이 "韓반일병 어이없다…야스쿠니 참배 당연" 5113
	[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 '적반하장' 일 우익 5130
	[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 '적반하장' 일 우익 5132
	[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 '적반하장' 일 우익 5142
	[이슈ON] 군함도 이어 사도광산도 뒤통수…'뒷북' 유감 표명 5154
	한일 외교장관, 오늘 이탈리아서 만날 듯…사도광산 논의하나 5168
	"한국 '반일병'에 신물…추도식 불참 항의해야" 日 극우신문 적반하장 사설 5177
[Cluster ID: 1]
	[속보] 김여정 "또 삐라…한국 것들 더러운 행위" 4813
	김여정 "선동삐라 또 떨어져…한국것들의 더러운 행위 강력히 규탄" 4817
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4823
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4824
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4825
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4826
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4827
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4828
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4829
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4830
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4831
	북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져" 4832
	김여정 "삐라 또 떨어져···한국 것들 강력 규탄" 4834
	

'[Cluster ID: 0]\n\t외교 2차관 "사도광산 추도식 불참이 강한 항의이자 유감 표명"\n\t한일 외교수장, G7계기 伊서 오늘 만날 듯…사도광산 논의\n\t외교부 "추도식 불참 통보 때 유감 표명과 항의"…무대응 비판에 뒤늦게 수습\n\t외교2차관 "사도광산 추도식 불참, 日에 강한 항의"(종합)\n\t\'우익 성향\' 日산케이 "韓반일병 어이없다…야스쿠니 참배 당연"\n\t[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 \'적반하장\' 일 우익\n\t[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 \'적반하장\' 일 우익\n\t[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 \'적반하장\' 일 우익\n\t[이슈ON] 군함도 이어 사도광산도 뒤통수…\'뒷북\' 유감 표명\n\t한일 외교장관, 오늘 이탈리아서 만날 듯…사도광산 논의하나\n\t"한국 \'반일병\'에 신물…추도식 불참 항의해야" 日 극우신문 적반하장 사설\n[Cluster ID: 1]\n\t[속보] 김여정 "또 삐라…한국 것들 더러운 행위"\n\t김여정 "선동삐라 또 떨어져…한국것들의 더러운 행위 강력히 규탄"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"\n\t김여정 "삐라 또 떨어져···한국 것들 강력 규탄"\n\t김여정 "삐라 또 떨어져…한국 것들 더러운 행위 강력 규탄"\n\t김여정 "南 삐라 또 떨어졌다…강력 규탄

In [6]:
clusters = {0: [4002, 3970, 4003, 3998, 3999],
 1: [4065, 4034, 4013, 4440, 4027, 4092, 4031],
 2: [4009, 4018, 4020, 4007],
 3: [4035,
  4037,
  4039,
  4060,
  4047,
  4087,
  4115,
  4149,
  4054,
  4086,
  4088,
  4091,
  4028,
  4062],
 4: [4609, 4677, 4744, 4691, 4057, 4670],
 5: [4064, 4066, 4099, 4068, 4070, 4143, 4050, 4114, 4093],
 6: [4096, 4101, 4102, 4104, 4094, 4095],
 7: [4610,
  4739,
  4743,
  4681,
  4682,
  4714,
  4750,
  4751,
  4656,
  4465,
  4694,
  4119,
  4761,
  4698,
  4731,
  4668,
  4734],
 8: [4132, 4164, 4165, 4142, 4081, 4467, 4148, 3990, 4159],
 9: [4224,
  4227,
  4229,
  4231,
  4232,
  4245,
  4253,
  4263,
  4267,
  4275,
  4277,
  4282,
  4286,
  4289,
  4292,
  4294,
  4169,
  4298,
  4172,
  4174,
  4176,
  4304,
  4306,
  4307,
  4309,
  4182,
  4310,
  4312,
  4186,
  4314,
  4315,
  4317,
  4190,
  4318,
  4192,
  4320,
  4196,
  4197,
  4326,
  4199,
  4201,
  4202,
  4205,
  4334,
  4210,
  4212,
  4215,
  4219,
  4220,
  4221],
 10: [4576, 4644, 4581, 4654, 4625, 4213, 4630, 4601, 4252, 4636],
 11: [4321, 4322, 4323, 4324, 4325, 4327],
 12: [4184, 4783, 4339, 4351],
 13: [4388, 4389, 4390, 4391, 4392, 4393, 4394, 4396, 4397, 4398, 4399],
 14: [4498, 4499, 4500, 4501, 4502, 4503, 4504, 4505, 4506],
 15: [4672, 4675, 4613, 4647, 4683, 4655, 4594, 4690, 4667, 4634, 4635, 4637],
 16: [4658, 4266, 4188, 4542],
 17: [4697, 4674, 4661, 4710]}

In [ ]:
result_string = \
"""
[Cluster ID: 0]
	[속보] 정부 "24일 사도광산 추도식 불참"···야스쿠니 참배 인사 참석 논란돼
	[속보] 정부, 사도광산 추도식 불참키로…日대표 야스쿠니 참배이력 문제
	정부, 일본 사도광산 추도식 하루 앞두고 전격 불참
	[속보] 정부 "사도광산 추도식 불참하기로 결정"
	정부, 사도광산 추도식 하루 전 '불참' 발표
[Cluster ID: 1]
	해병대, '연평도 포격전' 14주기 추모행사…"헌신·희생 기억"
	연평도 포격전 14주년…아들 보낸 부모는 오늘도 울었다
	연평도 포격전 14주년…"전투영웅 희생 잊지 않겠다"
	해병대, 연평도 포격전 14주년 전승기념식 거행
	연평도 포격전 14주년 전승기념식…"영웅 헌신·희생 기억"
	연평도 포격 14주년 행사…"영웅들 희생 기억"
	해병대, 연평도 포격전 14주년 전승기념식 개최…"영웅 헌신·희생 기억"
[Cluster ID: 2]
	[속보] 정부, 사도광산 추도식 불참키로…일 대표 야스쿠니 참배이력 문제
	[속보] 정부, 사도광산 추도식 불참 결정…일본대표 '야스쿠니 참배' 이력 문제
	[속보] 정부, 사도광산 추도식 불참키로…日대표 야스쿠니 참배이력 문제
	[속보] 정부, 사도광산 추도식 불참키로…日대표 야스쿠니 참배이력 문제
[Cluster ID: 3]
	野 "'파우치 박' 임명 강행…김건희 방송국으로 전락"
	민주 "박장범 사장 임명, KBS '김건희 방송' 전락"
	민주, 박장범 KBS 사장 임명안 재가에 "김건희 방송사 전락"
	민주당, 박장범 KBS 사장 임명에 "김건희 방송국으로 전락"
	민주, 박장범 KBS사장 임명에 "김건희 방송국으로 전락"
	민주, 박장범 KBS 사장 임명안 재가에 "김건희 방송국으로 전락"
	민주당, 박장범 사장 임명에 "KBS를 '김건희 방송국'으로 전락"
	민주, 박장범 KBS 사장 임명안 재가에 "김건희 방송국으로 전락" 비판
	민주당, 박장범 KBS 사장 임명안 재가에 "김건희 방송사로 전락"
	민주당, 박장범 사장 임명에 "KBS를 '김건희 방송국'으로 전락시켜"
	민주, 박장범 KBS 사장 임명안 재가에 "김건희 방송국으로 전락"
	野, 박장범 KBS 사장 임명 강행에 "김건희 방송사로 전락"
	민주 "윤, 박장범 임명 강행…인사청문회 신경 안 써"
	野 "尹, 결국 '파우치박' KBS사장 임명…아첨언론 새지평"
[Cluster ID: 4]
	"美 노동계 아메리칸드림 부활"…트럼프, 차베스-디레머 노동장관 지명
	트럼프, 노동장관에 차베스-디레머 지명…온건파 초선의원(종합)
	트럼프, 노동장관에 초선 하원의원 차베스-디레머 지명
	트럼프, 노동부 장관에 로리 차베스 드레머 하원의원 지명
	트럼프, 노동부 장관에 차베스-디레머 지명…"아메리칸드림 부활"
	트럼프, 1기 대북협상 실무자 알렉스 웡 국가안보부보좌관 지명
[Cluster ID: 5]
	여당 "이재명, 사법부 신뢰한다면 겁박 시위 멈춰야"
	국민의힘 "이재명, 진정 사법부 신뢰한다면 법원 겁박 시위 멈춰라"
	與 "이재명 사법부 신뢰한다면 '법원 겁박 시위'부터 멈춰야"
	與 "이재명, 사법부 신뢰한다면 법원 겁박 시위 멈춰야"
	與 "이재명 법원 겁박 시위부터 멈춰라"
	與 "민주당 장외집회 '아버지 이재명 구하기'…법원 겁박 시위"
	與 "민주당 시위는 이재명 방탄 위한 법원 겁박용"
	오늘 민주당 국정농단 규탄 집회에…與 "법원 겁박 시위 멈추길"
	"법원 겁박 시위 중단하라"…국힘, 이재명 직격
[Cluster ID: 6]
	김계환 사령관, 연평도 포격전 제14주년 기념행사 기념사
	'연평도 포격전 제14주년 전투영웅 추모 및 전승기념행사'
	김계환 사령관, 연평도 포격전 제14주년 기념행사 기념사
	'전사자들의 숭고한 희생을 영원히 잊지 않겠습니다'
	'영웅들을 추모하며'
	김계환 사령관, 연평도 포격전 제14주년 기념행사 기념사
[Cluster ID: 7]
	트럼프, '北협상 경험' 알렉스 웡 발탁…"中견제·北대화 등 다목적 카드"
	트럼프, NSC부보좌관에 알렉스 웡 지명…"북미 정상회담 기여"
	트럼프, 국가안보부보좌관에 '北 협상 경험' 알렉스 웡 지명
	北美대화 신호탄?…트럼프, 국가안보 부보좌관에 '정상회담 실무' 웡(종합)
	트럼프, 백악관 안보직에 1기 대북협상가 발탁…북미대화 신호?
	북미회담 실무자 '국가안보부보좌관' 지명…트럼프, 대북협상 재시동?
	트럼프, 백악관 안보직에 1기 대북협상가 발탁…북미대화 신호?(종합)
	'머그샷'까지 찍었던 트럼프의 선례…대통령 되면 재판 중단되나 [뉴스+]
	[속보] 美 CNN "북한군, 우크라 국경 넘었다…마리우폴·하르키우에도 도착"
	"파병 안했다"는 북한, 포로 발생 시 '살인죄' 처벌될 수
	"트럼프, 집권 2기 재무장관에 헤지펀드사 창업자 베센트 지명"
	트럼프 당선 최대 수혜 '비트코인', 10만달러선 눈앞에
	"트럼프, '경제 사령탑' 재무장관에 헤지펀드 CEO 베센트 지명"(상보)
	북한, 미국 전략 자산 한반도 전개에 "파국 몰아넣을 수 있는 발단"
	트럼프 NSC부보좌관에 '북핵통' 알렉스 웡 지명
	트럼프, 국가안보부보좌관에 알렉스 웡…북미정상회담 실무자
	러 "'오레시니크'의 모든 탄두, 목표물에 도달"
[Cluster ID: 8]
	윤 대통령, 박장범 KBS 사장 임명안 재가
	[속보] 尹대통령, 박장범 KBS 사장 임명안 재가
	윤 대통령, 박장범 KBS 사장 임명안 재가
	윤석열 대통령, 박장범 KBS 사장 임명 재가
	尹대통령, 박장범 KBS 사장 임명안 재가
	윤석열 대통령, 박장범 KBS 사장 임명 재가
	윤석열 대통령, 박장범 KBS 사장 임명안 재가
	윤석열 대통령, 박장범 KBS 사장 임명안 재가
	尹대통령, 박장범 KBS 사장 임명안 재가
[Cluster ID: 9]
	인천 만수동 만수 담방마을 아파트 49㎡ 1억5500만원에 거래
	성남 이매동 이매촌한신 84㎡ 13억3000만원에 거래
	인천 청라동 한양수자인레이크블루 아파트 84㎡ 7억7000만원에 거래
	인천 가정동 루원시티프라디움아파트 85㎡ 6억2500만원에 거래
	수원 매탄동 매탄주공4단지 73㎡ 8억2000만원에 거래
	용인 상현동 진산마을성원상떼빌아파트 59㎡ 5억4700만원에 거래
	인천 송도동 송도더샵그린스퀘어 98㎡ 8억4000만원에 거래
	용인 풍덕천동 현대성우 59㎡ 7억5000만원에 거래
	고양 장항동 장항호수마을2단지현대 70㎡ 5억1000만원에 거래
	고양 장항동 장항호수마을2단지현대 84㎡ 6억3000만원에 거래
	용인 상현동 상현엘지자이 84㎡ 6억원에 거래
	용인 신갈동 신흥덕 롯데캐슬레이시티 59㎡ 5억5500만원에 거래
	성남 야탑동 야탑매화마을주공2단지 58㎡ 6억9300만원에 거래
	용인 언남동 장미마을 삼성래미안2차 84㎡ 6억5000만원에 거래
	성남 서현동 서현시범한양 35㎡ 8억1000만원에 거래
	수원 이의동 광교자연앤힐스테이트 84㎡ 15억원에 거래
	성남 서현동 서현시범한양 148㎡ 21억원에 거래
	인천 송도동 더샵센트럴시티아파트 59㎡ 6억4200만원에 거래
	수원 조원동 수원한일타운아파트 59㎡ 4억4500만원에 거래
	인천 송도동 더샵센트럴시티아파트 72㎡ 6억9000만원에 거래
	성남 창곡동 위례호반써밋에비뉴 98㎡ 13억1000만원에 거래
	서울 용강동 래미안마포리버웰 84㎡ 22억5000만원에 거래
	서울 금호동4가 서울숲2차푸르지오 84㎡ 18억4500만원에 거래
	서울 명일동 명일지에스 84㎡ 9억4000만원에 거래
	서울 명일동 명일지에스 84㎡ 9억2500만원에 거래
	서울 도화동 도화현대홈타운 113㎡ 13억7000만원에 거래
	서울 도곡동 도곡삼성래미안 84㎡ 25억원에 거래
	서울 옥수동 옥수극동 142㎡ 16억5000만원에 거래
	서울 장지동 송파더센트레아파트 51㎡ 9억8000만원에 거래
	서울 서초동 서초래미안 111㎡ 24억원에 거래
	서울 성산동 성산시영아파트 59㎡ 11억5000만원에 거래
	서울 반포동 반포미도아파트 84㎡ 27억9000만원에 거래
	서울 역삼동 역삼래미안 59㎡ 22억7000만원에 거래
	서울 신월동 신월시영아파트 50㎡ 5억3500만원에 거래
	서울 옥수동 옥수파크힐스아파트 84㎡ 20억7000만원에 거래
	서울 역삼동 역삼래미안 59㎡ 21억8000만원에 거래
	서울 도곡동 도곡렉슬 114㎡ 38억4000만원에 거래
	서울 방이동 올림픽선수기자촌아파트 83㎡ 21억원에 거래
	서울 잠실동 잠실레이크팰리스 135㎡ 28억5000만원에 거래
	서울 압구정동 압구정신현대 152㎡ 71억원에 거래
	서울 대치동 래미안 대치 팰리스 94㎡ 42억2000만원에 거래
	서울 잠실동 잠실아시아선수촌 151㎡ 41억7000만원에 거래
	서울 잠실동 잠실5단지아파트 76㎡ 28억800만원에 거래
	서울 잠실동 잠실5단지아파트 76㎡ 28억2700만원에 거래
	서울 잠실동 잠실5단지아파트 81㎡ 30억4590만원에 거래
	부산 화명동 화명롯데캐슬카이저 84㎡ 5억9500만원에 거래
	서울 잠실동 잠실동트리지움 84㎡ 24억8500만원에 거래
	서울 압구정동 압구정신현대 108㎡ 50억원에 거래
	서울 대치동 대치미도맨션 126㎡ 39억원에 거래
	서울 대치동 대치미도맨션 126㎡ 40억원에 거래
[Cluster ID: 10]
	'380㎏ 에메랄드' 값이 1조4055억 달하는데…사람들 부르는 별명이
	1.4조 '저주받은 에메랄드'…23년 만에 고향으로
	1조4000억 가치 '저주받은 에메랄드' 23년만에 고향 가나
	1조4000억 '저주받은 에메랄드'…결국 23년 만에 브라질 가나
	'저주받은 에메랄드' 23년 만에 고향 브라질로 돌아갈 가능성
	1.4조 원 '저주받은 에메랄드' 23년 만에 고향 브라질행
	"23년 만에 고향으로"···1조4000억 가치 '저주받은 에메랄드'의 사연
	1조4000억짜리 초거대 '저주받은 에메랄드' 23년만에 고향으로
	'저주받은 에메랄드' 23년 만에 고향 브라질로 간다
	1조4천억 가치 '저주받은 에메랄드' 23년 만에 고향 브라질로
[Cluster ID: 11]
	'한-중 문화·관광장관 회담'
	'한-중 문화·관광장관 회담'
	'한-중 문화·관광장관 회담'
	'한-중 문화·관광장관 회담'
	'한-중 문화·관광장관 회담'
	'한-중 문화·관광장관 회담'
[Cluster ID: 12]
	기름값 6주 연속 상승…"다음 주 소폭 하락"
	휘발유·경유 가격 6주 연속 상승…러시아·우크라이나 전쟁 격화 등 영향
	"놀러가기 무섭네" 국내 주유소 기름값 6주 연속 상승
	[위클리 스마트] '취향저격' 핫플 여기 있었네…네이버 '히든 아카이브'
[Cluster ID: 13]
	손팻말 들고 구호 외치는 집회 참가자들
	손팻말 들고 구호 외치는 집회 참가자들
	손팻말 들고 구호 외치는 집회 참가자들
	주말 도심 집회
	주말 도심 집회
	손팻말 들고 구호 외치는 집회 참가자들
	세종대로에서 열린 집회
	세종대로에서 열린 집회
	손팻말 들고 구호 외치는 집회 참가자들
	손팻말 들고 구호 외치는 집회 참가자들
	손팻말 들고 구호 외치는 집회 참가자들
[Cluster ID: 14]
	노동자의 외침
	국회 앞 외침
	'국회는 일을 하라'
	'일을 해라'
	국회 앞 공공운수노조의 외침
	'22대 국회는 할 일을 하라'
	'일 좀 하세요'
	엄길용 위원장 '거부권 신기록 세우고 있는 대통령'
	공공운수노조의 외침
[Cluster ID: 15]
	트럼프, 주택도시장관에 NFL선수 출신 발탁…첫 흑인 장관 후보
	트럼프, 집권 2기 주택도시장관에 'NFL 선수 출신' 터너 발탁
	트럼프, 주택도시장관에 'NFL선수' 출신 스콧 터너 지명
	트럼프 2기 첫 흑인 장관은 스콧 터너…NFL 스타출신 주택도시장관 발탁
	트럼프, 주택도시장관에 '미식축구 선수 출신' 스콧 터너 지명
	트럼프, 주택도시장관에 NFL선수 출신 지명…첫 흑인 장관 후보
	트럼프, 주택도시장관에 NFL선수 출신 발탁…첫 흑인 장관 후보
	트럼프, 주택도시장관에 NFL선수 출신 발탁…첫 흑인 장관 후보
	트럼프, 주택도시장관에 미식축구 선수 출신  '스콧 터너' 지명
	트럼프, 주택도시개발부 장관에 '스콧 터너' 지명…"전 미식축구 선수"
	트럼프, 주택도시개발부 장관에 스콧 터너 지명…첫 흑인 후보
	트럼프, 주택도시장관에 NFL선수 출신 발탁···첫 흑인 장관 후보
[Cluster ID: 16]
	무디스, 사우디 신용등급 상향…2016년 이후 처음
	무디스, 사우디 신용등급 Aa3로 상향…평가 이래 처음
	무디스, 사우디 국가 신용등급 상향 조정…평가 이래 처음
	무디스, 사우디 신용등급 첫 상향…전망도 '긍정적'으로 올려
[Cluster ID: 17]
	CNN "북한군, 마리우폴·하르키우에도 도착"
	"북한군, 마리우폴·하르키우서 목격"…투입 전선 확대되나
	北, 한미일 연합훈련에 반발…"파국 몰아넣을 수 있는 발단"
	러 파병 북한군, 국경 넘었나…"우크라서 포착"
"""

In [ ]:
print(result_string)

# 7. 짜집기 뉴스 제작

## 7.1. (수작업) 가장 잘 군집화 된 군집 선택

In [18]:
from wrapper.llm_wrapper import LLM

llm = LLM()

llm.__del__()
llm.__init__(n_ctx=14000)

llm.set_prompt(
    f"""
    [요구 사항]
    다음은 군집별 뉴스 기사의 제목입니다.
    제목을 보고, 가장 유사한 항목끼리 잘 묶인 군집의 Cluster ID 값을 3개만 찾아 쉼표로 구분하여 한 줄로 출력하십시오.

    [제약 사항]
    다른 수식어구 없이 해당 Cluster의 ID 값만을 말씀하십시오.
    """
)
result = llm.generate(result_string)
result

/home/gpp/.local/lib/python3.11/site-packages/llama_cpp/llama.py:1138: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


'0, 1, 3, 5, 6'

In [19]:
top_clusters = [int(i) for i in result.split(",")]

In [20]:
from wrapper.llm_wrapper import LLM
from entity.entity import Article
from tqdm import tqdm


llm.__del__()
llm.__init__(max_tokens=8192, temperature=0.6, top_p=0.7)


def news_creation_chain(llm: LLM, news_contents: str) -> str:
    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 요약들을 종합하여 15개의 문장으로 정리해 주세요.

        [준수 사항]
        각 문장은 ~했어요, ~해요로 종결되는 300자 내외의 완결된 문장이어야 합니다.
        핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어야 합니다.


        [참고 사항]
        하나의 뉴스 요약에서, 첫 번째 문장은 이 기사에서 다루는 핵심 사건, 두 번째 문장은 사건의 배경과 관련된 맥락, 세 번째 문장은 사건의 진행과 결과를 설명합니다.
        """
    )
    processed = llm.generate(news_contents)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 초고를 완결된 3문단 구성의 뉴스 기사로 만들어 주세요.

        [준수 사항]
        뉴스 기사는 고등학생들이 읽을 것이에요. 친근한 말투 부탁해요.
        뉴스 기사는 3문단으로 구성되어야 해요.
        뉴스 기사의 모든 문장은 친근한 말투인 ~했어요, ~해요로 종결되어야 해요.
        뉴스 기사의 모든 문장은 300자 내외의 완결된 문장이어야 해요.

        [참고 사항]
        이 초고는 핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어 있어요.
        """
    )
    processed = llm.generate(processed)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 문장들에 대하여, 모든 문장을 ~했어요, ~해요, ~어요로 종결되게 수정해 주세요.

        [예시]
        최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있습니다. -> 최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있어요.
        """
    )

    result = llm.generate(processed)
    return result

finals: list[Article] = []
for cluster in tqdm(top_clusters):
    llm.__del__()
    llm.__init__()
    news_contents: list = []

    for news_id in clusters[cluster]:
        target_news = get_summerized_news(news_id)
        print(target_news.title)
        news_contents.append(target_news.content)

    result = news_creation_chain(llm, '\n\n'.join(i[2:] for i in news_contents))

    llm.__del__()
    llm.__init__()

    llm.set_prompt(
        """
        [요청 사항]
        다음 Article에 대한 제목을 지어 주세요.

        [제약 사항]
        제목은 40자 내외의 간결한 문장이어야 합니다.
        """
        )

    title = llm.generate(
        f"""
        {result}
        """
    )

    finals.append(Article(title=title, content=result, news_id=clusters[cluster][:]))


  0%|          | 0/5 [00:00<?, ?it/s]

외교부 "추도식 불참 통보 때 유감 표명과 항의"…무대응 비판에 뒤늦게 수습
[이슈ON] 군함도 이어 사도광산도 뒤통수…'뒷북' 유감 표명
"한국 '반일병'에 신물…추도식 불참 항의해야" 日 극우신문 적반하장 사설
[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 '적반하장' 일 우익
[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 '적반하장' 일 우익
외교 2차관 "사도광산 추도식 불참이 강한 항의이자 유감 표명"
한일 외교장관, 오늘 이탈리아서 만날 듯…사도광산 논의하나
한일 외교수장, G7계기 伊서 오늘 만날 듯…사도광산 논의
[영상] "한국 반일병 지긋지긋"…사도광산 갈등 속 '적반하장' 일 우익
'우익 성향' 日산케이 "韓반일병 어이없다…야스쿠니 참배 당연"
외교2차관 "사도광산 추도식 불참, 日에 강한 항의"(종합)


 20%|██        | 1/5 [00:42<02:48, 42.08s/it]

북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
김여정 "삐라 또 떨어져···한국 것들 강력 규탄"
김여정 "삐라 또 떨어져…한국 것들 더러운 행위 강력 규탄"
[속보] 김여정 "또 삐라…한국 것들 더러운 행위"
김여정 "선동삐라 또 떨어져…한국것들의 더러운 행위 강력히 규탄"
김여정 "南 삐라 또 떨어졌다…강력 규탄"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"
북한, "남쪽 국경선 부근에 정치선동 삐라 또 떨어져"


 40%|████      | 2/5 [01:24<02:06, 42.15s/it]

...

 60%|██████    | 3/5 [02:13<01:30, 45.31s/it]

[속보] 신임 대법관 최종후보 마용주…대통령에 제청
[속보] 신임 대법관 후보에 마용주 서울고법 부장판사 제청
[속보] 신임 대법관 최종후보 마용주…대통령에 제청
신임 대법관 최종후보 마용주 서울고법 부장판사
신임 대법관 최종후보 마용주…대통령에 제청
신임 대법관 최종 후보 마용주…대통령에 제청
신임 대법관 최종후보 마용주…대통령에 제청
'대법관 임명제청' 마용주 서울고법 부장판사는 누구(종합)


 80%|████████  | 4/5 [02:56<00:44, 44.27s/it]

[속보] 尹대통령 '김여사 특검법' 재의요구안 재가
민주 "사는 길은 '김건희 특검' 수용뿐" 압박…재의결 시기 저울질
김여사 특검법 재의요구권 행사에 여야 공방 격화
尹대통령, '김건희 여사 특검법' 세 번째 거부권 행사
[속보] 여야, 내달 10일 김건희 여사 특검법 재의결하기로
이젠 25번째…尹대통령의 '거부권 딜레마'
윤 대통령, 세 번째 '김건희 여사 특검법' 거부권 행사···김 여사 문제엔 '버티기' 일관
윤, '김건희 특검법' 3번째 거부…재의요구 법안 25개째
'김건희 특검법' 거부권 건의에 민주당 "거짓말 골프나 치는 대통령 정상 아냐"
윤 대통령, 세 번째 '김 여사 특검법' 재의요구안 재가
민주 초선들 "이재명 기소는 폭거…어떤 난관도 돌파해 정권 교체"
다시 국회 돌아간 '김건희 특검법'…野, 재표결 추진
尹, 김건희 특검법 올해만 3번째 '거부권 행사'


100%|██████████| 5/5 [03:36<00:00, 43.20s/it]


In [21]:
finals

[Article(title='"사도광산 추도식 불참, 한국-일본 외교 갈등 심화"', content='한국, 사도광산 추도식 불참에 일본 비판, 외교부 수습 나선\n\n한국 정부는 사도광산 추도식에 불참하기로 결정했으며, 일본 측의 무대응에 대해 비판이 제기되자 수습에 나섰습니다. 사도광산은 17~19세기 에도 시대에 일본에서 가장 많은 금을 생산한 곳으로, 강제 노역이 이루어진 역사적 장소입니다. 한국 정부는 사도광산의 유네스코 등재를 반대했으나, 약속된 조건을 일본이 충족하지 않아 불참을 결정했습니다.\n\n한국 정부는 추도식 불참을 통해 일본의 과거사 사과에 대한 진정성을 요구했으나, 산케이 신문은 이를 무시하고 일본 정치인의 야스쿠니 신사 참배를 강력히 지지했습니다. 사도광산 갈등의 배경에는 한국과 일본 간의 역사적 갈등과 야스쿠니 신사 참배 문제가 있으며, 일본의 우익 성향이 이를 심화시키고 있습니다.\n\n한국 정부는 일본의 야스쿠니 신사 참배와 추도식 관련 협의 과정에서 일본의 진정성을 보이지 않았다고 판단해 불참 결정에 이르렀습니다. 일본 측은 야스쿠니 신사 참배를 통해 전몰자에 대한 존경의 뜻을 표현하는 것이 당연하다고 주장하며, 이를 통해 한국 정부의 반일 감정을 비판했습니다. 외교부 2차관은 사도광산 추도식에 대한 한국의 불참이 일본에 대한 강한 항의이자 유감 표명이라고 밝혔습니다.', news_id=[4833, 5154, 5177, 5130, 5132, 4816, 5168, 4819, 5142, 5113, 4895]),
 Article(title='북한, 한국의 정치선동 삐라 도발에 강력 반발', content='북한이 한국의 정치선동 삐라 도발에 강력 반발했어요.\n\n북한은 최근 남쪽 국경선 부근에 한국이 보낸 정치선동 삐라가 또 떨어졌다고 주장하며 강력한 규탄을 가했어요. 북한은 한국이 남쪽 국경선 부근에 정치선동 삐라를 날려 보내는 행위를 반공화국 정치모략선동물로 규정하며 강력히 비난하고 있어요. 북한은 이러한 도발을 감행한 한국의

In [23]:
from wrapper.api_wrapper import ApiWrapper

api = ApiWrapper()

In [24]:
import json


article_indexes = []

for article in finals:
    result = api.upload_article(article)
    print(result.status_code)
    print(json.loads(result.content.decode())["data"][0]["articleIdx"])
    article_indexes.append(json.loads(result.content.decode())["data"][0]["articleIdx"])


200
41
200
42
200
43
200
44
200
45


## 퀴즈 생성

In [93]:
TARGET_ARTICLE = 3

In [97]:
article = finals[TARGET_ARTICLE]

ones, twos, threes = [], [], []

for i in article.news_id:
    one, two, three = get_summerized_news(i).content.split('\n')
    ones.append(one.split('.')[1].strip())
    twos.append(two.split('.')[1].strip())
    threes.append(three.split('.')[1].strip())


llm.__del__()
llm.__init__(temperature=0.7, top_p=0.8)

llm.set_prompt(
    f"""
    [요청 사항]
    다음 뉴스 기사로부터 80자 이내의 완결된 문장을 하나 만들어 주세요.

    [제약 사항]
    문장은 완결체로 종결되어야 합니다.
    문장에 해당 뉴스 기사의 핵심 요지가 담겨있어야 합니다.

    [예시]
    미국과 일본이 체결한 안보 협력 강화 협정이 공식 발효됨에 따라, 양국 군대의 인도-태평양 지역 합동 훈련이 본격화될 전망이다.
    """
)

result = llm.generate(
    # '\n'.join(ones)
    # + '\n'.join(twos)
    # + '\n'.join(threes)
    article.content
)

result
# result = '윤석열 대통령이 박장범을 KBS 사장으로 임명하자, 더불어민주당은 "KBS를 \'김건희 방송사\'로 전락시켰다"고 강하게 비판했어요.'

/home/gpp/.local/lib/python3.11/site-packages/llama_cpp/llama.py:1138: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


'윤석열 대통령이 대법관 후임자로 마용주 서울고법 부장판사를 임명제청했어요.'

Article(title='"운전자가 차량 들이받고 도주, 12시간 만에 자진 출석"', content="지난 23일 16시 경 광주 광산구 아파트 주차장에서 차량 4대를 들이받고 벤츠를 버린 운전자가 사건 발생 12시간 만에 경찰에 자진 출석했다. 사고 직후 운전자는 아무런 조치 없이 연락처와 벤츠를 남기고 도주했으며, 사고 12시간 40분 만에 같은 날 오후 6시쯤 경찰에 출석했다. 경찰은 A씨를 들이받은 차량을 수습하지 않은 채 도주한 혐의로 조사하고 있으며, CCTV를 통해 운전 경로를 추적 수사할 방침이다.\n\n동덕여대는 본관 점거 학생들을 강제 퇴거와 업무 방해 가처분 신청을 서울북부지법에 낼 예정이다. 동덕여대는 '남녀공학 전환' 논의 철회를 요구하며 본관 점거를 계속하고 있으며, 학교와 학생들이 서로 양보할 수 없는 입장차로 논의가 결렬되었다. 동덕여대는 본관 점거 학생들을 강제 퇴거와 업무 방해 가처분 신청을 서울북부지법에 낼 예정이다.\n\n한 빌라의 주차장에서 1년간 민폐 주차가 이어지자, 주민이 결국 차량을 똑같이 세워보았고, 상대 차주는 차를 바짝 붙여 차를 뺄 수 없도록 했다. 창원시 양곡동 가고초등학교 앞 버스정류장에서 시내버스 두 대가 추돌해 승객 17명이 경상을 입고 병원으로 이송됐다. 이 사고는 오후 2시 7분에 발생했으며, 경찰은 운전자들을 상대로 정확한 사고 원인을 조사 중이다. 경찰은 CCTV 등을 통해 운전 경로를 추적 수사하고 있으며, 운전자들이 사고 직후 도주한 사실이 확인되면서 강력한 조치를 취하고 있다.", news_id=[4845, 4883, 5107, 5081, 4892])

In [ ]:
# result = '한국 정부는 사도광산 추도식 불참을 통해 일본의 과거사 사과에 대한 진정성을 요구했으나, 일본은 이를 무시하고 일본 정치인들의 야스쿠니 신사 참배를 지지하며 갈등을 심화시키고 있다.'

In [98]:
llm.__del__()
llm.__init__(temperature=0.6, top_p=0.9)

llm.set_prompt(
    f"""
    [지시 사항]
    아래 문장에서 첫 번째 절을 추출하십시오.
    첫 번째 절은 문장의 앞부분으로, 주어와 서술어를 포함하며, 이후 부사구, 형용사구, 종속절 등이 시작되기 전까지의 부분입니다.

    [예시]
    입력: 미국과 일본이 체결한 안보 협력 강화 협정이 공식 발효됨에 따라 양국 군대의 인도-태평양 지역 합동 훈련이 본격화될 전망이다.
    출력: 미국과 일본이 체결한 안보 협력 강화 협정이 공식 발효됨에 따라
    """
)

quiz = llm.generate(
    # '\n'.join(ones)
    # + '\n'.join(twos)
    # + '\n'.join(threes)
    result
).replace("**", "*")

quiz

/home/gpp/.local/lib/python3.11/site-packages/llama_cpp/llama.py:1138: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


'윤석열 대통령이 대법관 후임자로 마용주 서울고법 부장판사를 임명제청했어요.'

In [99]:
quiz = "윤석열 대통령이 대법관 후임자로"

In [100]:
keyword = result[len(quiz):]
print(keyword)

quiz_result = quiz + ' ' + ("_" * len(keyword))
quiz_result

 마용주 서울고법 부장판사를 임명제청했어요.


'윤석열 대통령이 대법관 후임자로 ________________________'

In [101]:
llm.__del__()
llm.__init__(temperature=0.7, top_p=0.7)

llm.set_prompt(
    f"""
    [요청 사항]
    이 문장을 완성시킨 결과를 출력해 주세요.

    [힌트]
    '_'의 수는 답의 글자 수입니다.
    """
    # [제약 사항]
    # 모든 문장은 ~했어요, ~해요로 종결되어야 합니다.
    # """
)

blank1 = llm.generate(
    quiz + keyword.split()[0]
)


blank2 = llm.generate(
    quiz + keyword.split()[0]
)

blank3 = llm.generate(
    quiz + keyword.split()[0]
)

print(blank1, '\n', blank2, '\n', blank3)

blanks = [blank1, blank2, blank3]


윤석열 대통령이 대법관 후임자로 명시했다. 
 윤석열 대통령이 대법관 후임자로 마용주를 지명했다. 
 윤석열 대통령이 대법관 후임자로 문용주를 지명했습니다.


In [102]:
blanks = [
    "마용주 서울고법 차장판사를 임명했어요.",
    "마용주 부산고법 부장판사를 임명제청했어요.",
    "마용주 진주고법 부장판사를 임명제청했어요."
]


In [103]:
result = api.send(
    endpoint="/quiz",
    method="POST",
    auth=True,
    data={
        "articleIdx": article_indexes[TARGET_ARTICLE],
        "quizContent": quiz_result,
        "answers": [
            {
                "answerContent": b,
                "answerYn": False
            } for b in blanks
        ]
        + [{
            "answerContent": keyword,
            "answerYn": True
        }]
    },
    query_str=False
)
print(result.status_code)

200


In [104]:
print(result.content.decode())

{"result":true,"data":[{"quizIdx":10,"articleIdx":44,"quizContent":"윤석열 대통령이 대법관 후임자로 ________________________","answers":[{"answerIdx":32,"answerContent":"마용주 서울고법 차장판사를 임명했어요.","answerYn":false},{"answerIdx":33,"answerContent":"마용주 부산고법 부장판사를 임명제청했어요.","answerYn":false},{"answerIdx":34,"answerContent":"마용주 진주고법 부장판사를 임명제청했어요.","answerYn":false},{"answerIdx":35,"answerContent":" 마용주 서울고법 부장판사를 임명제청했어요.","answerYn":true}]}],"resultCount":1}


In [76]:
print(
    api.send(
        endpoint="/quiz",
        method="GET",
        auth=True
    ).content.decode()
)

{"result":true,"data":[[{"quizIdx":2,"articleIdx":15,"quizContent":"여기서 질문! 무엇을 먹자고 했을까요?","answers":[{"answerIdx":1,"answerContent":"토스트","answerYn":false},{"answerIdx":2,"answerContent":"국밥","answerYn":true},{"answerIdx":3,"answerContent":"아이스크립","answerYn":false}]},{"quizIdx":3,"articleIdx":36,"quizContent":"일본의 야스쿠니 신사 참배 이력 문제로 인해,  __________________________","answers":[{"answerIdx":4,"answerContent":"일본의 야스쿠니 신사 참배 이력 문제로 인해, 정부는 대대적인 사과와 사과식당 설치에 나섰어요.","answerYn":false},{"answerIdx":5,"answerContent":"일본의 야스쿠니 신사 참배 이력 문제로 인해, 정부는 강한 비판을 받았어요.","answerYn":false},{"answerIdx":6,"answerContent":"일본의 야스쿠니 신사 참배 이력 문제로 인해, 정부는 공식적인 사과와 사과금을 제공했어요.","answerYn":false},{"answerIdx":7,"answerContent":"정부는 사도광산 추도식에 불참하기로 결정했어요.","answerYn":true}]},{"quizIdx":4,"articleIdx":36,"quizContent":"해병대가 23일 대전 유성구 국립대전현충원에서 연평도 포격전 14주년을 기념하는  ________________________","answers":[{"answerIdx":8,"answerContent":"해병대가 23일 대전 유성구 국립대전현충원에서 연평도 포격전 14주년을 기념하는 전투영웅을 추모했어요.","answerYn":false},{"ans